# 🧠 Step 9 — TCN Architecture with Deep Fusion FNN

## What changed from previous version

**Numerical branch:** Linear Transformer → **Temporal Convolutional Network (TCN)**
- TCN uses dilated 1D convolutions to capture patterns at multiple time scales simultaneously
- Dilation rates: 1, 2, 4, 8 → captures 1-day, 2-day, 4-day, 8-day momentum patterns
- Faster to train than transformer, better for short sequences (≤20 days)
- No positional encoding needed — convolutions are inherently position-aware

**Fusion FNN:** Shallow (96→64→1) → **Deep with residual connections (96→256→128→64→32→1)**
- Residual skip connections prevent vanishing gradients in deep layers
- More capacity to learn complex interactions between price and sentiment signals

**Everything else unchanged:**
- Same 5 input files
- Same FinBERT textual branch
- Same Huber loss + Adam optimiser
- Same train/test split

## Files needed (5)
1. `nifty50_step4_standard_train.csv`
2. `nifty50_step4_standard_test.csv`
3. `step7b_finbert_embeddings.csv`
4. `step7a_finbert_sentiment_comparison.csv`
5. `standard_scaler.pkl`

---
**Enable GPU:** Runtime → Change runtime type → T4 GPU

Then: **Runtime → Run All**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────────────────────────────────────
!pip install torch pandas numpy scikit-learn joblib -q
print('✓ Libraries ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Imports and Configuration
# ─────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import joblib
import os
import math
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ Device: {device}')
if str(device) == 'cpu':
    print('  ⚠️  No GPU — enable via Runtime → Change runtime type → T4 GPU')

# ── MODEL CONFIG ──────────────────────────────────────────────────────────
SEQUENCE_LENGTH  = 20     # days of history
TCN_CHANNELS     = 64     # TCN output channels
TCN_KERNEL_SIZE  = 3      # convolution kernel size
TCN_DILATIONS    = [1, 2, 4, 8]  # dilation rates — captures 1,2,4,8 day patterns
TCN_DROPOUT      = 0.2
TEXT_PROJ_DIM    = 64     # FinBERT 773 → 64
FUSION_DIMS      = [256, 128, 64, 32]  # deep FNN with residuals
DROPOUT          = 0.2
LEARNING_RATE    = 3e-5
EPOCHS           = 200
BATCH_SIZE       = 16
PATIENCE         = 25
HUBER_DELTA      = 0.5
WARMUP_EPOCHS    = 10

print(f'\n  Architecture: TCN + Deep Fusion FNN')
print(f'  TCN channels   : {TCN_CHANNELS}')
print(f'  TCN dilations  : {TCN_DILATIONS}  (captures 1,2,4,8 day patterns)')
print(f'  TCN kernel     : {TCN_KERNEL_SIZE}')
print(f'  Text proj dim  : {TEXT_PROJ_DIM}')
print(f'  Fusion FNN     : {TCN_CHANNELS} + {TEXT_PROJ_DIM} → {" → ".join(map(str,FUSION_DIMS))} → 1')
print(f'  Sequence length: {SEQUENCE_LENGTH} days')
print(f'  Loss           : Huber (delta={HUBER_DELTA})')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — Upload and Load All 5 Input Files
# ─────────────────────────────────────────────────────────────────────────

print('Upload 1/5: nifty50_step4_standard_train.csv')
u = files.upload(); train_num_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(train_num_df)} train rows')

print('\nUpload 2/5: nifty50_step4_standard_test.csv')
u = files.upload(); test_num_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(test_num_df)} test rows')

print('\nUpload 3/5: step7b_finbert_embeddings.csv')
u = files.upload(); emb_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(emb_df)} embedding rows')

print('\nUpload 4/5: step7a_finbert_sentiment_comparison.csv')
u = files.upload(); sent_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(sent_df)} sentiment rows')

print('\nUpload 5/5: standard_scaler.pkl')
u = files.upload(); scaler = joblib.load(list(u.keys())[0])
print(f'  ✓ Scaler loaded')

# ── Parse dates ────────────────────────────────────────────────────────────
for df in [train_num_df, test_num_df]:
    df['date'] = pd.to_datetime(
        df['Date'], dayfirst=True, errors='coerce'
    ).dt.strftime('%Y-%m-%d')

emb_df['date'] = pd.to_datetime(
    emb_df['date'], dayfirst=True, errors='coerce'
).dt.strftime('%Y-%m-%d')

sent_df['date'] = pd.to_datetime(
    sent_df['date'], dayfirst=True, errors='coerce'
).dt.strftime('%Y-%m-%d')
sent_df = sent_df.dropna(subset=['date'])
for col in ['finbert_score','gpt4o_score',
            'finbert_pos_prob','finbert_neg_prob','finbert_neu_prob']:
    if col in sent_df.columns:
        sent_df[col] = pd.to_numeric(sent_df[col], errors='coerce').fillna(0)

# ── Feature columns ────────────────────────────────────────────────────────
EXCLUDE_COLS  = ['Date', 'date', 'Close', 'LogReturn_Close']
NUM_FEAT_COLS = [c for c in train_num_df.columns if c not in EXCLUDE_COLS]
TARGET_COL    = 'LogReturn_Close'
EMB_COLS      = [c for c in emb_df.columns if c.startswith('emb_')]
SENT_COLS     = [c for c in ['finbert_score','gpt4o_score',
                              'finbert_pos_prob','finbert_neg_prob',
                              'finbert_neu_prob'] if c in sent_df.columns]

print(f'\n  Numerical features : {len(NUM_FEAT_COLS)}')
print(f'  Embedding dims     : {len(EMB_COLS)}')
print(f'  Sentiment features : {SENT_COLS}')
print(f'  Train dates        : {train_num_df["date"].iloc[0]} → {train_num_df["date"].iloc[-1]}')
print(f'  Test dates         : {test_num_df["date"].iloc[0]} → {test_num_df["date"].iloc[-1]}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Merge Features
# ─────────────────────────────────────────────────────────────────────────

def merge_features(num_df, emb_df, sent_df, num_cols, emb_cols, sent_cols, target_col):
    cols_needed = ['date', target_col] + [c for c in num_cols if c != target_col]
    merged = pd.merge(num_df[cols_needed],
                      emb_df[['date'] + emb_cols],
                      on='date', how='left')
    if sent_cols:
        merged = pd.merge(merged,
                          sent_df[['date'] + sent_cols],
                          on='date', how='left')
        merged[sent_cols] = merged[sent_cols].fillna(0)
    merged[emb_cols] = merged[emb_cols].fillna(0)
    merged = merged.dropna(subset=num_cols).reset_index(drop=True)
    return merged

train_merged = merge_features(train_num_df, emb_df, sent_df,
                               NUM_FEAT_COLS, EMB_COLS, SENT_COLS, TARGET_COL)
test_merged  = merge_features(test_num_df,  emb_df, sent_df,
                               NUM_FEAT_COLS, EMB_COLS, SENT_COLS, TARGET_COL)

train_emb_cov = (train_merged[EMB_COLS[0]] != 0).mean() * 100
test_emb_cov  = (test_merged[EMB_COLS[0]]  != 0).mean() * 100

print(f'  Train rows         : {len(train_merged)}')
print(f'  Test rows          : {len(test_merged)}')
print(f'  Train emb coverage : {train_emb_cov:.1f}%')
print(f'  Test emb coverage  : {test_emb_cov:.1f}%')
print(f'  Target stats       : mean={train_merged[TARGET_COL].mean():.4f}  std={train_merged[TARGET_COL].std():.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — Build Sequence Dataset
# ─────────────────────────────────────────────────────────────────────────

class StockDataset(Dataset):
    def __init__(self, df, num_cols, emb_cols, sent_cols, target_col, seq_len):
        self.seq_len  = seq_len
        self.num_data = df[num_cols].values.astype(np.float32)
        all_txt       = emb_cols + sent_cols
        self.txt_data = df[all_txt].values.astype(np.float32)
        self.targets  = df[target_col].values.astype(np.float32)
        self.n        = len(df) - seq_len

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        num_seq = self.num_data[idx : idx + self.seq_len]  # (seq_len, num_feats)
        txt_vec = self.txt_data[idx + self.seq_len - 1]    # last day text
        target  = self.targets[idx + self.seq_len]          # next day target
        return (
            torch.tensor(num_seq, dtype=torch.float32),
            torch.tensor(txt_vec, dtype=torch.float32),
            torch.tensor(target,  dtype=torch.float32),
        )

train_dataset = StockDataset(
    train_merged, NUM_FEAT_COLS, EMB_COLS, SENT_COLS, TARGET_COL, SEQUENCE_LENGTH
)
test_dataset  = StockDataset(
    test_merged,  NUM_FEAT_COLS, EMB_COLS, SENT_COLS, TARGET_COL, SEQUENCE_LENGTH
)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

NUM_FEATURES  = len(NUM_FEAT_COLS)
TXT_DIM       = len(EMB_COLS) + len(SENT_COLS)

print(f'  Train sequences  : {len(train_dataset)}')
print(f'  Test sequences   : {len(test_dataset)}')
print(f'  Num features     : {NUM_FEATURES}')
print(f'  Text vector dim  : {TXT_DIM}')

nb, tb, lb = next(iter(train_loader))
print(f'  Batch shapes:')
print(f'    Numerical : {nb.shape}')
print(f'    Text vec  : {tb.shape}')
print(f'    Target    : {lb.shape}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Define TCN + Deep Fusion FNN Model
#
# TCN (Temporal Convolutional Network):
#   Uses dilated causal convolutions to capture patterns at multiple scales
#   Dilation 1 → sees 1-day patterns  (recent momentum)
#   Dilation 2 → sees 2-day patterns  (short-term trend)
#   Dilation 4 → sees 4-day patterns  (weekly pattern)
#   Dilation 8 → sees 8-day patterns  (bi-weekly pattern)
#
# Deep Fusion FNN with Residual Connections:
#   Residual = output + input (skip connection)
#   Prevents vanishing gradients in deep layers
#   Allows gradient to flow directly from loss to early layers
# ─────────────────────────────────────────────────────────────────────────

class TCNBlock(nn.Module):
    """Single TCN block with dilated causal convolution + residual connection."""
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()
        # Padding to maintain sequence length with causal convolution
        padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels, out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding
        )
        self.bn      = nn.BatchNorm1d(out_channels)
        self.gelu    = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        # Residual projection if channels differ
        self.residual = nn.Conv1d(in_channels, out_channels, 1) \
                        if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        # x: (batch, channels, seq_len)
        out = self.conv(x)
        # Remove future-looking padding (causal)
        out = out[:, :, :x.size(2)]
        out = self.bn(out)
        out = self.gelu(out)
        out = self.dropout(out)
        return out + self.residual(x)


class ResidualFNNBlock(nn.Module):
    """FNN block with residual skip connection."""
    def __init__(self, in_dim, out_dim, dropout):
        super().__init__()
        self.linear  = nn.Linear(in_dim, out_dim)
        self.bn      = nn.BatchNorm1d(out_dim)
        self.gelu    = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.residual = nn.Linear(in_dim, out_dim) \
                        if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        return self.dropout(self.gelu(self.bn(self.linear(x)))) + self.residual(x)


class TCNMultimodalPredictor(nn.Module):
    def __init__(self, num_features, txt_dim, tcn_channels, kernel_size,
                 dilations, tcn_dropout, text_proj_dim, fusion_dims, dropout):
        super().__init__()

        # ── Numerical branch: TCN ─────────────────────────────────────────
        # Input projection: num_features → tcn_channels
        self.input_proj = nn.Linear(num_features, tcn_channels)

        # Stack of TCN blocks with increasing dilation
        self.tcn_blocks = nn.ModuleList([
            TCNBlock(tcn_channels, tcn_channels, kernel_size, d, tcn_dropout)
            for d in dilations
        ])

        # ── Textual branch: FinBERT projection ───────────────────────────
        self.txt_proj = nn.Sequential(
            nn.Linear(txt_dim, text_proj_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(text_proj_dim * 2, text_proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # ── Deep Fusion FNN with residual connections ─────────────────────
        fusion_input_dim = tcn_channels + text_proj_dim
        fnn_layers       = []
        in_dim           = fusion_input_dim
        for out_dim in fusion_dims:
            fnn_layers.append(ResidualFNNBlock(in_dim, out_dim, dropout))
            in_dim = out_dim
        fnn_layers.append(nn.Linear(in_dim, 1))
        self.fusion_fnn = nn.Sequential(*fnn_layers)

    def forward(self, num_seq, txt_vec):
        # num_seq: (batch, seq_len, num_features)
        # txt_vec: (batch, txt_dim)

        # ── TCN branch ────────────────────────────────────────────────────
        x = self.input_proj(num_seq)     # (B, seq_len, tcn_channels)
        x = x.permute(0, 2, 1)          # (B, tcn_channels, seq_len) for Conv1d
        for block in self.tcn_blocks:
            x = block(x)
        x = x[:, :, -1]                 # last timestep: (B, tcn_channels)

        # ── Text branch ───────────────────────────────────────────────────
        t = self.txt_proj(txt_vec)       # (B, text_proj_dim)

        # ── Fusion ────────────────────────────────────────────────────────
        fused = torch.cat([x, t], dim=1) # (B, tcn_channels + text_proj_dim)
        out   = self.fusion_fnn(fused)   # (B, 1)
        return out.squeeze(-1)           # (B,)


model = TCNMultimodalPredictor(
    num_features  = NUM_FEATURES,
    txt_dim       = TXT_DIM,
    tcn_channels  = TCN_CHANNELS,
    kernel_size   = TCN_KERNEL_SIZE,
    dilations     = TCN_DILATIONS,
    tcn_dropout   = TCN_DROPOUT,
    text_proj_dim = TEXT_PROJ_DIM,
    fusion_dims   = FUSION_DIMS,
    dropout       = DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  TCN Multimodal Predictor')
print(f'  Total parameters: {total_params:,}')
print()
print(model)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — Train the Model
# ─────────────────────────────────────────────────────────────────────────

criterion  = nn.HuberLoss(delta=HUBER_DELTA)
optimizer  = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(WARMUP_EPOCHS)
    progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler      = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
best_val_loss  = float('inf')
patience_count = 0
best_epoch     = 0
train_losses   = []
val_losses     = []

print('=' * 60)
print('  TRAINING — TCN + Deep Fusion FNN')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):

    # Train
    model.train()
    train_loss = 0.0
    for num_seq, txt_vec, target in train_loader:
        num_seq = num_seq.to(device)
        txt_vec = txt_vec.to(device)
        target  = target.to(device)
        optimizer.zero_grad()
        pred   = model(num_seq, txt_vec)
        loss   = criterion(pred, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for num_seq, txt_vec, target in test_loader:
            num_seq = num_seq.to(device)
            txt_vec = txt_vec.to(device)
            target  = target.to(device)
            pred    = model(num_seq, txt_vec)
            loss    = criterion(pred, target)
            val_loss += loss.item()
    val_loss /= len(test_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step()

    if epoch % 10 == 0 or epoch == 1:
        lr = optimizer.param_groups[0]['lr']
        print(f'  Epoch {epoch:>3}/{EPOCHS}  '
              f'Train={train_loss:.6f}  '
              f'Val={val_loss:.6f}  '
              f'LR={lr:.2e}')

    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_epoch     = epoch
        patience_count = 0
        torch.save(model.state_dict(), 'step9_tcn_weights.pth')
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'\n  Early stopping at epoch {epoch}')
            print(f'  Best epoch    : {best_epoch}')
            print(f'  Best val loss : {best_val_loss:.6f}')
            break

print(f'\n✅ Training complete')
print(f'  Best epoch    : {best_epoch}')
print(f'  Best val loss : {best_val_loss:.6f}')
model.load_state_dict(torch.load('step9_tcn_weights.pth', map_location=device))
print('  Best weights loaded')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 8 — Evaluate
# ─────────────────────────────────────────────────────────────────────────

model.eval()
all_preds   = []
all_targets = []

with torch.no_grad():
    for num_seq, txt_vec, target in test_loader:
        num_seq = num_seq.to(device)
        txt_vec = txt_vec.to(device)
        pred    = model(num_seq, txt_vec).cpu().numpy()
        all_preds.extend(pred)
        all_targets.extend(target.numpy())

preds   = np.array(all_preds)
actuals = np.array(all_targets)

mae   = mean_absolute_error(actuals, preds)
rmse  = np.sqrt(mean_squared_error(actuals, preds))
r2    = r2_score(actuals, preds)

nonzero = np.abs(actuals) > 1e-4
mape    = np.mean(np.abs((actuals[nonzero]-preds[nonzero])/actuals[nonzero]))*100 \
          if nonzero.sum() > 0 else np.nan

dir_acc = np.mean(np.sign(actuals) == np.sign(preds)) * 100

print('=' * 65)
print('  EVALUATION — TCN + Deep Fusion FNN')
print('=' * 65)
print(f'  MAE                  : {mae:.6f}')
print(f'  RMSE                 : {rmse:.6f}')
print(f'  R²                   : {r2:.4f}')
print(f'  Directional Accuracy : {dir_acc:.2f}%')
print()
print('  ─────────────────────────────────────────────────')
print('  COMPARISON TABLE:')
print('  ─────────────────────────────────────────────────')
print(f'  Naive baseline                    : ~51.00%')
print(f'  Baseline LSTM (OHLC only, 11yr)   :  50.94%')
print(f'  Transformer + FinBERT (prev best) :  57.35%')
print(f'  TCN + Deep FNN (this run)         :  {dir_acc:.2f}%')
print()
improvement = dir_acc - 57.35
if improvement > 0:
    print(f'  ✅ TCN improved over Transformer by {improvement:+.2f}%')
else:
    print(f'  ⚠️  TCN did not improve over Transformer ({improvement:+.2f}%)')

# Save
metrics_df = pd.DataFrame([{
    'model':                'TCN + Deep Fusion FNN',
    'MAE':                  round(mae, 6),
    'RMSE':                 round(rmse, 6),
    'R2':                   round(r2, 4),
    'Directional_Accuracy': round(dir_acc, 2),
    'Best_Epoch':           best_epoch,
    'Best_Val_Loss':        round(best_val_loss, 6),
}])
metrics_df.to_csv('step9_tcn_metrics.csv', index=False)
print(f'\n  Metrics saved → step9_tcn_metrics.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 9 — Save Predictions and Download
# ─────────────────────────────────────────────────────────────────────────

test_dates = test_merged['date'].values[SEQUENCE_LENGTH:]
test_dates = test_dates[:len(preds)]

pred_df = pd.DataFrame({
    'date':              [pd.to_datetime(d).strftime('%d-%b-%Y') for d in test_dates],
    'actual_log_return': actuals,
    'pred_log_return':   preds,
    'actual_direction':  np.where(actuals >= 0, 'UP', 'DOWN'),
    'pred_direction':    np.where(preds   >= 0, 'UP', 'DOWN'),
    'direction_correct': np.sign(actuals) == np.sign(preds),
    'error':             np.abs(actuals - preds),
})
pred_df.to_csv('step9_tcn_predictions.csv', index=False)

loss_df = pd.DataFrame({
    'epoch':      list(range(1, len(train_losses)+1)),
    'train_loss': train_losses,
    'val_loss':   val_losses,
})
loss_df.to_csv('step9_tcn_training_loss.csv', index=False)

output_files = [
    ('step9_tcn_predictions.csv',    'Predicted vs actual'),
    ('step9_tcn_metrics.csv',        'MAE, RMSE, R², Directional Accuracy'),
    ('step9_tcn_training_loss.csv',  'Train/val loss per epoch'),
    ('step9_tcn_weights.pth',        'Saved model weights'),
]

print('=' * 60)
print('  TCN MODEL COMPLETE')
print('=' * 60)
for fname, desc in output_files:
    if os.path.exists(fname):
        print(f'  ✓ {fname}  ({os.path.getsize(fname)/1024:.1f} KB)')

print()
print(f'  Directional Accuracy : {dir_acc:.2f}%')
print(f'  Best epoch           : {best_epoch}')

print()
print('⬇️  Downloading...')
for fname, _ in output_files:
    if os.path.exists(fname):
        files.download(fname)
        print(f'  ✓ {fname}')

print()
print('✅ Done! Share the directional accuracy to compare with Transformer result.')